In [2]:
import os
os.listdir('/content/drive')

['MyDrive', 'Shareddrives', '.shortcut-targets-by-id', '.Trash-0']

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


transform = transforms.Compose([transforms.Resize((224, 224)),  transforms.ToTensor(), transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225] #standard resnet
    )])
dataset = datasets.ImageFolder(root="data/PlantVillage", transform=transform)
print(dataset.classes)


['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'PlantVillage', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [ ]:
import random
from torch.utils.data import DataLoader, Subset

#convert images to binary classifcation
new_target = []
#need to update both dataset target and sample for ImageFolder
for label in dataset.targets:
    name = dataset.classes[label]
    if "healthy" in name:
        binary_label = 0
    else:
        binary_label = 1

    new_target.append(binary_label)
dataset.targets = new_target

#sample
sample = []
for i in range(len(dataset.samples)):
    path = dataset.samples[i][0]
    label = new_target[i]
    sample.append((path, label))
dataset.samples = sample #append path and target label


dataset.classes = ["healthy", "diseased"]
dataset.class_to_idx = {"healthy": 0, "diseased": 1}
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

img, label = dataset[0]
print(label, img.shape)

#had to use smaller datset because it taking forever
random.seed(42)
small_indices = random.sample(range(len(dataset)), 3000)
dataset = Subset(dataset, small_indices)
dataset.targets = [dataset.dataset.targets[i] for i in small_indices]
dataset.classes = ["healthy", "diseased"]

1 torch.Size([3, 224, 224])


In [5]:
pip install tensorflow

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

X = []
Y = []

#convert tensor to numpy for test train
for i in range(len(dataset)):
    imagetensor, label = dataset[i]
    tonumpy = imagetensor.numpy()
    nowimagenumpy = np.transpose(tonumpy, (1, 2, 0))
    X.append(nowimagenumpy)
    Y.append(label)

#convert to numpy
X = np.array(X)
Y = np.array(Y)


X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import KFold
import numpy as np
import tensorflow as tf
import os
import gc


X = []
Y = []

#convert tensor to numpy for test train
for i in range(len(dataset)):
    imagetensor, label = dataset[i]
    tonumpy = imagetensor.numpy()
    nowimagenumpy = np.transpose(tonumpy, (1, 2, 0))
    X.append(nowimagenumpy)
    Y.append(label)

#convert to numpy
X = np.array(X)
Y = np.array(Y)

batch_size = 8

imagesize = (224, 224)

epochs = 2

results = []

num_folds = 5
#cross validation of 5 folds
kfold = KFold(n_splits=num_folds, shuffle=True, random_state=42)


foldoutput = []

for fold, (train, valdate) in enumerate(kfold.split(X_train)):

    #splits training data each time for each fold model
    X_fold_train = X_train[train]
    X_fold_output = X_train[valdate]

    Y_fold_train = Y_train[train]
    Y_fold_output = Y_train[valdate]


    traindatagen = tf.keras.preprocessing.image.ImageDataGenerator(rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
        shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
    )
    trainfold = traindatagen.flow(X_fold_train, Y_fold_train, batch_size=8, shuffle=True)

    valdatedatagen = tf.keras.preprocessing.image.ImageDataGenerator()
    valdatefold = valdatedatagen.flow(X_fold_output, Y_fold_output, batch_size=8, shuffle=False)

    restNet50 = ResNet50(input_shape = imagesize + (3,), weights='imagenet', include_top=False)

    #freeze weights
    for layer in restNet50.layers:
        layer.trainable = False

    classes = dataset.classes

    #add pooling layer
    pooling_layer = tf.keras.layers.GlobalAveragePooling2D()(restNet50.output)

    #attention mechanism
    attention_layer1 = tf.keras.layers.Dense(128, activation='sigmoid')(pooling_layer)

    attention_layer2 = tf.keras.layers.Dense(2048, activation='relu')(attention_layer1)

    attention = tf.keras.layers.Multiply()([pooling_layer, attention_layer2])

    #flatten it
    flattened = tf.keras.layers.Flatten()(attention)

    anotherlayer = tf.keras.layers.Dense(256, activation='relu')(flattened)

    norm = tf.keras.layers.BatchNormalization()(anotherlayer)

    dropout = tf.keras.layers.Dropout(0.5)(norm)

    finallayer = tf.keras.layers.Dense(1, activation='sigmoid')(dropout)

    model = tf.keras.Model(inputs=restNet50.input, outputs=finallayer)

    print(model.summary())

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])


    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint("best_model.h5", save_best_only=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, verbose=1)
    ]

    m = model.fit(trainfold, validation_data=valdatefold, epochs=epochs, callbacks=callbacks, steps_per_epoch=len(X_fold_train) // batch_size, validation_steps=len(X_fold_output) // batch_size)


    validation = max(m.history['val_accuracy'])
    training_accuracy = max(m.history['accuracy'])
    loss = m.history['loss'][-1]
    foldoutput.append({'Fold': fold + 1, 'Train Accuracy': training_accuracy, 'Validation Accuracy': validation, 'Loss': loss})

    plt.figure(figsize=(10, 6))
    plt.plot(m.history['accuracy'], label='train accuracy', linewidth=2)
    plt.plot(m.history['val_accuracy'], label='val accuracy', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Model Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f'training_history_fold_{fold + 1}.png', dpi=300, bbox_inches='tight')
    plt.show()

    results.append(model)

    del model, restNet50, trainfold, valdatefold
    gc.collect()


fold_output = []
for fold_result, model in enumerate(results):

    foldpredictions = model.predict(X_test, verbose=0)
    predictions_classification = (foldpredictions > 0.5).astype(int).flatten()
    accuracy = (predictions_classification == Y_test).mean()

    #csv for baysian
    predictions_df = pd.DataFrame({
        'Fold': fold_result + 1,
        'Actual Classification': Y_test,
        'Model Predicted Label': predictions_classification,
        'Prediction Probability': foldpredictions.flatten(),
        'Correct Classification Or Not': (predictions_classification == Y_test).astype(int)
    })

    predictions_df.to_csv(f'predictions_{fold_result + 1}fold.csv', index=False)


    fold_output.append({
        'Fold': fold_result + 1,
        'Test Accuracy': accuracy * 100
    })

cross_validation_df = pd.DataFrame(fold_output)
cross_validation_df.to_csv(f'Cross Validation results on {num_folds} folds.csv', index=False)


Fold 1/5
Train: (1920, 224, 224, 3) | Val: (480, 224, 224, 3)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,640,001 (93.99 MB)

 Trainable params: 1,051,777 (4.01 MB)

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>